# Preparação dos dados do Ensembl

Este notebook existe para preparar os arquivos mais recentes do Ensembl antes de qualquer etapa de treino ou avaliação.

Os dados vêm compactados, com nomes longos e sem padronização direta para uso no pipeline. Se usados assim, isso gera dificuldade para automatizar loops por espécie, aumenta chance de erro e dificulta manutenção do projeto.

Por isso, este notebook organiza essa base inicial.

## O que ele faz

O notebook executa as seguintes etapas:

1. **Configurações de ambiente**  
   Imports para o desenvolvido, centralização de todos os caminhos do projeto.

2. **Inspeção dos arquivos brutos**  
   Lista os arquivos baixados e identifica quais são `cds` e `ncrna`.

3. **Descompactação dos arquivos (`.fa.gz`)**  
   Converte os arquivos para `.fa`, deixando-os utilizáveis nas próximas etapas.

4. **Normalização dos nomes**  
   Simplifica os nomes dos arquivos para um padrão consistente por espécie e tipo (cds / ncrna).

5. **Organização da estrutura**  
   Garante que os arquivos estejam organizados de forma previsível para uso em scripts futuros.

6. **Validação dos dados**  
   Verifica se cada espécie possui os dois grupos esperados:
   - codificante (`cds`)
   - não codificante (`ncrna`)

   Verifica estatísticas sobre as sequencias:
   - Quantidade de sequencias
   - Menor cadeia
   - Maior cadeia

## Resultado esperado

Ao final, os dados estarão:
- descompactados  
- com nomes padronizados  
- organizados por espécie  
- prontos para uso no restante do pipeline

## 1. Configurações de ambiente

### 1.1 Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

### 1.2 Paths

In [2]:
# Pasta base do projeto RNAmining
BASE_RNAMINING = Path("/home/samuel/projects/RNAmining-updated")

# Pasta com os novos FASTAs do ensemble
S5_DIR = BASE_RNAMINING / "volumes" / "rnamining-front" / "data" / "S5_File"

# Pasta onde os arquivos organizados serão armazenados
ORG_DIR = S5_DIR / "Organisms_Sequences"

# Caminho do CSV que contem as estatísticas sobre as sequências
CSV_OUT = ORG_DIR / "organism_sequences_stats.csv"

In [3]:
# Cria a pasta caso não exista
ORG_DIR.mkdir(exist_ok=True)

### 2. Inspeção dos arquivos brutos

Checagem de quantos arquivos FASTAs compactados foram extraidos para a pasta

In [4]:
gz_files = list(S5_DIR.glob("*.gz"))

print(f"Total de arquivos .gz: {len(gz_files)}")

Total de arquivos .gz: 0


Dos arquivos, quantos são cds e quantos são ncRNA. Checado apenas para garantir que todas as espécies tiveram suas sequencias codificantes e não codificantes extraidas.

In [5]:
cds_files = [f for f in gz_files if "cds" in f.name]
ncrna_files = [f for f in gz_files if "ncrna" in f.name]

print(f"CDS: {len(cds_files)}")
print(f"ncRNA: {len(ncrna_files)}")

CDS: 0
ncRNA: 0


Exemplos dos arquivos extraidos.

In [6]:
print("CDS: ")
for f in cds_files[:5]:
    print(f.name)

print("\nncRNA: ")
for f in ncrna_files[:5]:
    print(f.name)

CDS: 

ncRNA: 


Ao inspecionar os nomes dos arquivos é possível identificar o possível padrão: Nome_da_especie.resto.(cds/ncrna).gz
A estratégia será salvar o nome de cada espécie, coletando o nome utilizando do padrão encontrado nos arquivos.

### 3. Descompactação dos arquivos (`.fa.gz`)

In [7]:
for file in S5_DIR.glob("*.fa.gz"):
    # Os arquivos extraidos são salvos com o seu nome sem o ".gz"
    output_file = file.with_suffix("")

    # Se ele já foi extraido, pula
    if output_file.exists():
        continue

    # Script do gunzip para descompactar
    !gunzip -c "{file}" > "{output_file}"
    

### 4. Normatização dos nomes

#### 4.1 Guardar os nomes de cada espécie

Será extraido o nome completo das espécies analisadas para a montagem de arquivos futuros, como: Nomes simplificados para facilitar a navegação nos dados, nomes de pastas .....

In [8]:
for file in S5_DIR.glob("*.fa"):
    # Os nomes das espécies estão antes do primeiro ponto
    especies = file.name.split(".")[0]

    # Checa qual é o tipo do arquivo em questão e guarda essa informação 
    if "cds" in file.name:
        tipo = "cds"
    elif "ncrna" in file.name:
        tipo = "ncrna"

    # Monta o novo nome
    novo_nome = f"{especies}.{tipo}.fa"

    # Print apenas pra acompanhar o processo
    print(file.name, "->", novo_nome)


    # Novo arquivo com nome atualizado
    novo_path = file.parent / novo_nome

    if novo_path.exists():
        continue

    # Subistitui o arquivo antigo com o novo
    file.rename(novo_path)


### 5. Organização da estrutura

In [9]:
for file in S5_DIR.glob("*.fa"):
    
    # Define o novo caminho do arquivo dentro da nova pasta
    new_path = ORG_DIR / file.name
    
    if new_path.exists():
        continue

    # Move o arquivo para a pasta criada
    file.rename(new_path)
    
    # Print apenas para acompanhar
    print(file.name, "->", new_path)

### 6. Validação dos dados

Fuções auxiliares para analisar as sequencias obitidas.

In [10]:
def get_species_and_type(file):
    """
    Extrai o nome da espécie e o tipo do arquivo a partir do nome.
    
    Ex:
    Homo_sapiens.cds.fa   -> ("Homo_sapiens", "cds")
    Homo_sapiens.ncrna.fa -> ("Homo_sapiens", "ncrna")
    """
    parts = file.name.split(".")
    species = parts[0]
    seq_type = parts[1]
    
    return species, seq_type


def get_fasta_lengths(fasta_path):
    """
    Lê um FASTA e retorna uma lista com os tamanhos das sequências.
    """
    lengths = []
    current_seq_len = 0

    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()

            # Se a linha começa com ">", começa uma nova sequência
            if line.startswith(">"):
                # Se já havia uma sequência sendo acumulada, salva o tamanho dela
                if current_seq_len > 0:
                    lengths.append(current_seq_len)
                    current_seq_len = 0
            else:
                # Soma o tamanho do trecho da sequência
                current_seq_len += len(line)

    # Adiciona a última sequência do arquivo
    if current_seq_len > 0:
        lengths.append(current_seq_len)

    return lengths


In [11]:
# Dicionário para organizar os arquivos por espécie
species_map = {}

for file in ORG_DIR.glob("*.fa"):

    # Extrai nome da espécie e tipo (cds ou ncrna)
    species, seq_type = get_species_and_type(file)

    # Se a espécie ainda não existe no dicionário, cria
    if species not in species_map:
        species_map[species] = {}

    # Associa o arquivo ao tipo correspondente da espécie
    species_map[species][seq_type] = file

In [12]:
# Looping que percorre cada espécie e seus arquivos associados
for species, files_dict in species_map.items():

    # Verifica se a espécie possui arquivo cds
    has_cds = "cds" in files_dict

    # Verifica se a espécie possui arquivo ncrna
    has_ncrna = "ncrna" in files_dict

    # Print para visualizar a integridade dos dados
    print(f"{species} -> cds: {has_cds} | ncrna: {has_ncrna}")

Chrysemys_picta_bellii -> cds: True | ncrna: True
Rattus_norvegicus -> cds: True | ncrna: True
Homo_sapiens -> cds: True | ncrna: True
Gallus_gallus -> cds: True | ncrna: True
Notechis_scutatus -> cds: True | ncrna: True
Eptatretus_burgeri -> cds: True | ncrna: True
Crocodylus_porosus -> cds: True | ncrna: True
Monodelphis_domestica -> cds: True | ncrna: True
Sphenodon_punctatus -> cds: True | ncrna: True
Ornithorhynchus_anatinus -> cds: True | ncrna: True
Anolis_carolinensis -> cds: True | ncrna: True
Xenopus_tropicalis -> cds: True | ncrna: True
Latimeria_chalumnae -> cds: True | ncrna: True
Mus_musculus -> cds: True | ncrna: True
Danio_rerio -> cds: True | ncrna: True
Petromyzon_marinus -> cds: True | ncrna: True


In [13]:
rows = []

for species, files_dict in species_map.items():

    # Checa se a espécie tem os dois grupos esperados
    has_cds = "cds" in files_dict
    has_ncrna = "ncrna" in files_dict

    for seq_type in ["cds", "ncrna"]:

        # Se não existir esse tipo para a espécie, salva como faltante
        if seq_type not in files_dict:
            rows.append({
                "species": species,
                "seq_type": seq_type,
                "has_cds": has_cds,
                "has_ncrna": has_ncrna,
                "file_name": None,
                "n_sequences": None,
                "min_len": None,
                "max_len": None,
                "mean_len": None,
                "median_len": None
            })
            continue

        file = files_dict[seq_type]
        lengths = get_fasta_lengths(file)

        # Caso o arquivo exista, mas venha vazio
        if len(lengths) == 0:
            rows.append({
                "species": species,
                "seq_type": seq_type,
                "has_cds": has_cds,
                "has_ncrna": has_ncrna,
                "file_name": file.name,
                "n_sequences": 0,
                "min_len": None,
                "max_len": None,
                "mean_len": None,
                "median_len": None
            })
            continue

        rows.append({
            "species": species,
            "seq_type": seq_type,
            "has_cds": has_cds,
            "has_ncrna": has_ncrna,
            "file_name": file.name,
            "n_sequences": len(lengths),
            "min_len": min(lengths),
            "max_len": max(lengths),
            "mean_len": np.mean(lengths),
            "median_len": np.median(lengths)
        })


In [14]:
stats_df = pd.DataFrame(rows)

# Ordena para ficar mais fácil de visualizar
stats_df = stats_df.sort_values(["species", "seq_type"]).reset_index(drop=True)

stats_df

,species,seq_type,has_cds,has_ncrna,file_name,n_sequences,min_len,max_len,mean_len,median_len
0,Anolis_carolinensis,cds,True,True,Anolis_carolinensis.cds.fa,32972,51,21030,2023.177059,1518.0
1,Anolis_carolinensis,ncrna,True,True,Anolis_carolinensis.ncrna.fa,4759,55,9669,1604.747636,1305.0
2,Chrysemys_picta_bellii,cds,True,True,Chrysemys_picta_bellii.cds.fa,39084,57,24105,1934.747953,1404.0
3,Chrysemys_picta_bellii,ncrna,True,True,Chrysemys_picta_bellii.ncrna.fa,7038,40,6284,709.371697,498.5
4,Crocodylus_porosus,cds,True,True,Crocodylus_porosus.cds.fa,27731,114,18873,2049.766326,1521.0
5,Crocodylus_porosus,ncrna,True,True,Crocodylus_porosus.ncrna.fa,4618,35,10601,566.962321,350.0
6,Danio_rerio,cds,True,True,Danio_rerio.cds.fa,52089,3,87459,1546.523162,1110.0
7,Danio_rerio,ncrna,True,True,Danio_rerio.ncrna.fa,8115,34,13525,645.162539,398.0
8,Eptatretus_burgeri,cds,True,True,Eptatretus_burgeri.cds.fa,27960,150,13806,1510.397997,1191.0
9,Eptatretus_burgeri,ncrna,True,True,Eptatretus_burgeri.ncrna.fa,1089,41,8965,1517.597796,1202.0


In [ ]:
stats_df.to_csv(CSV_OUT, index=False)
print(f"CSV salvo em: {CSV_OUT}")